In [ ]:
# ============================================================
# FLIGHT DATA 2024 — Dataset Evaluation & Source Preparation
# Google Colab Notebook
# ============================================================

# ── 0. Install & Imports ─────────────────────────────────────
!pip install openpyxl kaggle --quiet

import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from google.colab import drive, files

drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/FlightDW/sources')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = '/content/drive/MyDrive/FlightDW/raw/flight_data_2024.csv'


In [ ]:
# ── 1. Kaggle Dataset Download (run once) ────────────────────
from google.colab import userdata

def setup_kaggle():
    """Configure Kaggle credentials from Colab Secrets and download the dataset."""
    # Pull credentials from Colab Secrets (Secrets tab → KAGGLE_USERNAME, KAGGLE_TOKEN)
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_TOKEN')

    # Also write kaggle.json so the CLI picks it up
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    kaggle_json = json.dumps({
        'username': os.environ['KAGGLE_USERNAME'],
        'key':      os.environ['KAGGLE_KEY'],
    })
    with open('/root/.config/kaggle/kaggle.json', 'w') as f:
        f.write(kaggle_json)
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

    Path('/content/drive/MyDrive/FlightDW/raw').mkdir(parents=True, exist_ok=True)

    !kaggle datasets download -d hrishitpatil/flight-data-2024 \
        -p /content/drive/MyDrive/FlightDW/raw --unzip

    print("Download complete.")

setup_kaggle()


In [ ]:
# ── 2. Efficient Load with Optimised dtypes ──────────────────
DTYPE_MAP = {
    'year':                 'Int16',
    'month':                'Int8',
    'day_of_month':         'Int8',
    'day_of_week':          'Int8',
    'op_unique_carrier':    'category',
    'op_carrier_fl_num':    'Int32',
    'origin':               'category',
    'origin_city_name':     'category',
    'origin_state_nm':      'category',
    'dest':                 'category',
    'dest_city_name':       'category',
    'dest_state_nm':        'category',
    'crs_dep_time':         'Int16',
    'dep_time':             'float32',
    'dep_delay':            'float32',
    'taxi_out':             'float32',
    'wheels_off':           'float32',
    'wheels_on':            'float32',
    'taxi_in':              'float32',
    'crs_arr_time':         'Int16',
    'arr_time':             'float32',
    'arr_delay':            'float32',
    'cancelled':            'Int8',
    'cancellation_code':    'category',
    'diverted':             'Int8',
    'crs_elapsed_time':     'float32',
    'actual_elapsed_time':  'float32',
    'air_time':             'float32',
    'distance':             'float32',
    'carrier_delay':        'Int16',
    'weather_delay':        'Int16',
    'nas_delay':            'Int16',
    'security_delay':       'Int16',
    'late_aircraft_delay':  'Int16',
}

print("Loading dataset — this may take a few minutes for 7M rows...")
df = pd.read_csv(
    RAW_FILE,
    dtype=DTYPE_MAP,
    parse_dates=['fl_date'],
    low_memory=False
)
print(f"Loaded: {len(df):,} rows × {len(df.columns)} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# ── 3. Dataset Evaluation ────────────────────────────────────

def evaluate_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produce a comprehensive evaluation report for every column.
    Covers: dtype, null count/pct, unique count, min, max, sample values.
    """
    rows = []
    total = len(df)

    for col in df.columns:
        s = df[col]
        null_count = s.isna().sum()
        null_pct   = round(null_count / total * 100, 2)
        unique     = s.nunique(dropna=True)

        # Safe min/max (avoid crash on all-null or categorical)
        try:
            col_min = s.dropna().min()
            col_max = s.dropna().max()
        except TypeError:
            col_min = col_max = 'n/a'

        samples = s.dropna().unique()[:5].tolist()

        rows.append({
            'column':        col,
            'dtype':         str(s.dtype),
            'total_rows':    total,
            'null_count':    int(null_count),
            'null_pct':      null_pct,
            'completeness':  f"{100 - null_pct:.2f}%",
            'unique_values': int(unique),
            'min':           col_min,
            'max':           col_max,
            'sample_values': str(samples),
        })

    return pd.DataFrame(rows)


print("\n── Generating dataset evaluation report...")
eval_df = evaluate_dataset(df)

print(eval_df[['column','dtype','null_pct','unique_values','sample_values']].to_string(index=False))

eval_df.to_csv(OUTPUT_DIR / 'dataset_evaluation_report.csv', index=False)
print(f"\nFull report saved to: {OUTPUT_DIR / 'dataset_evaluation_report.csv'}")

In [ ]:
# ── 4. Data Quality Checks ───────────────────────────────────

def run_quality_checks(df: pd.DataFrame):
    """Run targeted business-logic quality checks."""
    issues = []

    # 4a. Cancelled flights should have null dep_time / arr_time
    cancelled = df[df['cancelled'] == 1]
    leak = cancelled['dep_time'].notna().sum()
    issues.append({'check': 'Cancelled flights with non-null dep_time',
                   'count': int(leak),
                   'severity': 'HIGH' if leak > 0 else 'OK'})

    # 4b. Non-cancelled flights should have dep_time
    operated = df[df['cancelled'] == 0]
    missing_dep = operated['dep_time'].isna().sum()
    issues.append({'check': 'Operated flights missing dep_time',
                   'count': int(missing_dep),
                   'severity': 'MEDIUM' if missing_dep > 0 else 'OK'})

    # 4c. Delay components only on non-cancelled, delayed flights
    delay_cols = ['carrier_delay','weather_delay','nas_delay',
                  'security_delay','late_aircraft_delay']
    cancelled_with_delay = cancelled[delay_cols].fillna(0).sum(axis=1)
    issues.append({'check': 'Cancelled flights with non-zero delay components',
                   'count': int((cancelled_with_delay > 0).sum()),
                   'severity': 'LOW'})

    # 4d. arr_delay vs dep_delay correlation sanity
    corr = df[['dep_delay','arr_delay']].dropna().corr().iloc[0,1]
    issues.append({'check': 'dep_delay ↔ arr_delay Pearson correlation',
                   'count': round(corr, 4),
                   'severity': 'OK' if corr > 0.8 else 'REVIEW'})

    # 4e. Impossible elapsed time (actual > 0 when cancelled)
    impossible = df[(df['cancelled'] == 1) & df['actual_elapsed_time'].notna()].shape[0]
    issues.append({'check': 'Cancelled flights with actual_elapsed_time',
                   'count': int(impossible),
                   'severity': 'HIGH' if impossible > 0 else 'OK'})

    # 4f. Date range coverage
    issues.append({'check': 'Earliest fl_date', 'count': str(df['fl_date'].min()), 'severity': 'INFO'})
    issues.append({'check': 'Latest fl_date',   'count': str(df['fl_date'].max()), 'severity': 'INFO'})
    issues.append({'check': 'Months covered',   'count': df['month'].nunique(),     'severity': 'INFO'})

    qc_df = pd.DataFrame(issues)
    print("\n── Quality Check Results:")
    print(qc_df.to_string(index=False))
    qc_df.to_csv(OUTPUT_DIR / 'quality_check_results.csv', index=False)
    return qc_df


qc_results = run_quality_checks(df)

In [ ]:
# ── 5. Unique Value Profiles for Lookup Columns ──────────────

def profile_lookup_columns(df: pd.DataFrame):
    """Detailed unique value breakdown for categorical columns."""
    lookup_cols = ['op_unique_carrier','origin','dest',
                   'origin_state_nm','dest_state_nm','cancellation_code',
                   'day_of_week','month']
    for col in lookup_cols:
        vc = df[col].value_counts(dropna=False).reset_index()
        vc.columns = ['value', 'count']
        vc['pct'] = (vc['count'] / len(df) * 100).round(2)
        print(f"\n── {col} ({vc.shape[0]} unique values):")
        print(vc.head(10).to_string(index=False))
        vc.to_csv(OUTPUT_DIR / f'profile_{col}.csv', index=False)


profile_lookup_columns(df)